# 03b · Extended fine-tuning (optional experiment 3)

**Why:** in notebook 03 the fine-tuned MobileNetV2 was still improving when it hit its
15-epoch limit: validation loss reached its lowest value (0.403) on the very last epoch.
This notebook continues training the same model for up to 20 more epochs, with the same
tiny learning rate and early stopping, and saves it as `mobilenetv2_finetuned_long.keras`.
Notebook 04 then compares it with the others on the validation set and keeps the winner.

**Run on Colab with a T4 GPU.** About 5–8 minutes.

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
# Colab only: copy the images to Colab's fast local disk (Drive is slow for 8,000 files)
if IN_COLAB:
    cfg = use_local_copy(cfg)

In [ ]:
import time
import pandas as pd
import tensorflow as tf
from src.data_loading import load_all_splits
from src.models import make_callbacks, count_params
from src.evaluate import plot_history

print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow)")
tf.keras.utils.set_random_seed(cfg["seed"])
T = cfg["training"]
MODELS = cfg["paths"]["models"]; METRICS = cfg["paths"]["metrics"]; FIG = cfg["paths"]["figures"]
train_ds, val_ds, test_ds = load_all_splits(cfg)   # expect 4,800 / 1,600 / 1,599

In [ ]:
model = tf.keras.models.load_model(MODELS / "mobilenetv2_finetuned.keras")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=T["finetune_lr"]),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(count_params(model))   # should show about 1.67 million trainable parameters

done = len(pd.read_csv(METRICS / "history_mobilenetv2.csv"))   # epochs already trained (head + fine-tune)
log = METRICS / "history_mobilenetv2_long.csv"; log.unlink(missing_ok=True)

t0 = time.time()
model.fit(train_ds, validation_data=val_ds, epochs=done + 20, initial_epoch=done,
          callbacks=make_callbacks(MODELS / "mobilenetv2_finetuned_long.keras", log, T["patience"]))
print(f"Training time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
# One continuous curve: head training + fine-tuning (notebook 03) + this extension
full = pd.concat([pd.read_csv(METRICS / "history_mobilenetv2.csv"), pd.read_csv(log)], ignore_index=True)
full.to_csv(METRICS / "history_mobilenetv2_full.csv", index=False)
plot_history(METRICS / "history_mobilenetv2_full.csv",
             f"MobileNetV2: head (1-15), fine-tune (16-{done}), extended ({done + 1}+)",
             FIG / "03b_history_mobilenetv2_full.png");

best = tf.keras.models.load_model(MODELS / "mobilenetv2_finetuned_long.keras")
tr_loss, tr_acc = best.evaluate(train_ds, verbose=0)
va_loss, va_acc = best.evaluate(val_ds, verbose=0)
row = {"model": "mobilenetv2_finetuned_long", "train_acc": round(tr_acc, 4), "val_acc": round(va_acc, 4),
       "gap": round(tr_acc - va_acc, 4), "val_loss": round(va_loss, 4), **count_params(best)}
summary = pd.read_csv(METRICS / "val_summary.csv")
summary = pd.concat([summary[summary["model"] != row["model"]], pd.DataFrame([row])], ignore_index=True)
summary.to_csv(METRICS / "val_summary.csv", index=False)
summary